MODULE
1. Encoder
2. Token_Representation
3. Entity FFN
4. Span FFN
5. Matching
6. Loss
7. Decder

In [ ]:
BATCH_SIZE=32
SPAN_CHANNEL=100

In [2]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer,AutoModel

In [3]:
#Encoder
BackBone_Model="klue/bert-base"

In [4]:
# Input -> (BATCH_SIZE,SEQ_LEN) , Each -> (SEQ_LEN,)
# Output -> (BATCH_SIZE,SEQ_LEN,Hidden_state)
class EncoderModule(nn.Module):
    def __init__(self,backbone_model:str):
        super().__init__()
        self.encoder=AutoModel.from_pretrained(backbone_model)
        self.hidden_size=self.encoder.config.hidden_size
    
    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor)->torch.Tensor:
        outputs=self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_states=outputs.last_hidden_state
        
        return hidden_states

In [5]:
# Token_Representation(h generator)
# Input -> (BATCH_SIZE,SEQ_LEN,Hidden_size) , (BATCH_SIZE,SEQ_LEN)
# Output -> (BATCH_SIZE,T_MAX,Hidden_size)
class Text_Token_Represntation(nn.Module):
    def __init__(self):
        super().__init__()
        
    def pad_sequence(self,sentences):
        max_len=max(line.size(0) for line in sentences)
        hidden_size=sentences[0].size(1)
        
        pad_sequences=[]
        
        for sent in sentences:
            pad_len=max_len-sent.size(0)
            
            if pad_len>0:
                pad_tensor=torch.zeros(
                    pad_len,
                    hidden_size,
                    device=sent.device,
                    dtype=sent.dtype
                )
                sent=torch.cat([sent,pad_tensor],dim=0)
                
            pad_sequences.append(sent)
        return torch.stack(pad_sequences,dim=0)
        
    def forward(self,hidden_states,text_mask):
        text_vector=[]
        
        for hs,mask in zip(hidden_states,text_mask):
            h_vector=hs[mask.bool()]
            text_vector.append(h_vector)
        
        padded_text_vectors=self.pad_sequence(text_vector)    
    
        return padded_text_vectors

In [6]:
# Test
hidden_states=torch.randn(2,6,4)
text_mask=torch.Tensor([
    [0,0,1,1,1,0],
    [0,1,1,0,0,0]
])
model=Text_Token_Represntation()
output=model(hidden_states,text_mask)


print("hidden_state shape : ",hidden_states.shape)
print("text_mask shape : ",text_mask.shape)
print("output shape : ",output.shape)
print(hidden_states)
#print("\n")
print(output)


hidden_state shape :  torch.Size([2, 6, 4])
text_mask shape :  torch.Size([2, 6])
output shape :  torch.Size([2, 3, 4])
tensor([[[-0.0375,  0.6288, -0.3618,  0.4050],
         [-0.8798, -1.6447,  0.1445,  0.4934],
         [ 1.5555, -0.1700, -0.3012,  0.3837],
         [ 0.8099,  0.1266,  2.0908,  0.2231],
         [ 0.0320,  0.5455,  0.6715, -2.3695],
         [ 0.1986, -0.3584,  1.9095,  0.4284]],

        [[ 0.1517,  0.7531,  0.1141,  1.2777],
         [-0.2206, -0.7930,  0.8735, -1.1392],
         [-0.4794, -0.7631, -1.0402,  0.2776],
         [-0.6403,  0.2442,  1.9649, -1.0488],
         [ 1.2836, -0.2090,  1.2694, -0.4572],
         [ 0.8001, -2.0768,  0.6363,  0.0312]]])
tensor([[[ 1.5555, -0.1700, -0.3012,  0.3837],
         [ 0.8099,  0.1266,  2.0908,  0.2231],
         [ 0.0320,  0.5455,  0.6715, -2.3695]],

        [[-0.2206, -0.7930,  0.8735, -1.1392],
         [-0.4794, -0.7631, -1.0402,  0.2776],
         [ 0.0000,  0.0000,  0.0000,  0.0000]]])


In [7]:
# Token_Representation(p generator)
# Input -> (BATCH_SIZE,SEQ_LEN,Hidden_size) , (BATCH_SIZE,SEQ_LEN)
# Output -> (BATCH_SIZE,M,Hidden_size) (M=[ENT] count)
class ENT_Token_Representation(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self,hidden_states,ent_mask):
        ent_vectors=[]
        
        for hs,mask in zip(hidden_states,ent_mask):
            p_vector=hs[mask.bool()]
            ent_vectors.append(p_vector)
            
        ent_vectors=torch.stack(ent_vectors,dim=0)
        
        return ent_vectors

In [8]:
# Test
hidden_states=torch.randn(2,8,4)
ent_token_mask=torch.tensor([
    [0,1,0,1,0,1,0,0],
    [0,1,0,1,0,1,0,0]
])
model=ENT_Token_Representation()
output=model(hidden_states,ent_token_mask)

print("hidden_states shape : ",hidden_states.shape)
print(hidden_states)
print("ent_token_mask shape : ",ent_token_mask.shape)
print("output shape : ",output.shape)
print(output)

hidden_states shape :  torch.Size([2, 8, 4])
tensor([[[-1.6864, -0.4198,  0.0166,  0.2023],
         [-1.1465, -1.0214, -0.5445,  0.8727],
         [ 0.5268,  1.0641,  1.8452, -1.5885],
         [-0.9077, -0.8177, -0.6570,  0.5584],
         [-0.8518,  0.2134,  0.1638,  0.7189],
         [-0.4497,  0.5367,  0.1814,  0.7158],
         [-1.2395,  0.1675,  1.3301, -1.0527],
         [ 0.3292, -0.3160,  0.4031,  0.0860]],

        [[-0.3196,  0.6765,  1.2850,  1.6596],
         [ 1.1823,  0.5228,  0.5768,  1.5740],
         [ 1.7832, -0.8014,  0.9666,  0.3100],
         [ 0.1121,  1.2046,  0.7465, -1.1225],
         [-0.5019, -0.8450, -1.3000,  1.2640],
         [ 0.1666, -1.5899, -0.1620,  1.3769],
         [-0.3705, -0.2825, -0.2724,  0.9176],
         [-0.0306, -0.9064, -1.1208,  1.5168]]])
ent_token_mask shape :  torch.Size([2, 8])
output shape :  torch.Size([2, 3, 4])
tensor([[[-1.1465, -1.0214, -0.5445,  0.8727],
         [-0.9077, -0.8177, -0.6570,  0.5584],
         [-0.4497,  0.53

In [9]:
class Linear(nn.Module):
    def __init__(self,input_x,output_y,bias=True):
        super().__init__()
        
        self.w=nn.Parameter(torch.randn(output_y,input_x))
        
        if bias:
            self.bias=nn.Parameter(torch.randn(output_y))
        else:
            self.bias=None
    def forward(self,x):
        out=x @ self.w.T
        
        if self.bias is not None:
            out+=self.bias
        return out

In [10]:
# test
linear=Linear(768,256)
x=torch.randn(32,10,768)
y=linear(x)
print(y.shape)

torch.Size([32, 10, 256])


In [11]:
#class ReLU(nn.Module):
#    def __init__(self):
#        super().__init__()
#    def forward(self,x):
#        if(x>=0):
#            return x
#        else:
#            return 0
# 이거는 Tensor 각각을 보지않아서 동작안함
class ReLU(nn.Module):
    def forward(self,x):
        return x*(x>0)

In [12]:
# test
activation=ReLU()
print(activation(-2))

0


In [13]:
class Sequential(nn.Module):
    def __init__(self,*layers):
        super().__init__()
        self.layers=nn.ModuleList(layers)
        
    def forward(self,x):
        for layer in self.layers:
            x=layer(x)

        return x

In [14]:
# test
model=Sequential(
    Linear(4,8),
    ReLU(),
    #Linear(8,2)
)
x=torch.randn(3,4)
y=model(x)
print(y.shape)
print(x)
print(y)

torch.Size([3, 8])
tensor([[-0.6330, -0.2072,  0.2726,  1.0812],
        [ 0.6703,  1.2264,  0.1800,  0.8323],
        [-0.1554, -0.5109,  0.2398,  1.9311]])
tensor([[-0.0000, -0.0000, -0.0000, -0.0000, 1.6419, 1.1014, 1.5985, -0.0000],
        [-0.0000, -0.0000, -0.0000, -0.0000, 0.8155, -0.0000, -0.0000, -0.0000],
        [-0.0000, -0.0000, -0.0000, -0.0000, 2.5549, 1.7977, -0.0000, -0.0000]],
       grad_fn=<MulBackward0>)


In [15]:
# Entity FFN
# input -> (Batch_size,M,d_model)
# output -> (Batch_size,M,d_model)
class Entity_Representation(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()
        
        self.net=Sequential(
            Linear(d_model,d_ff),
            ReLU(),
            Linear(d_ff,d_model)
        )
    def forward(self,x):
        output=self.net(x)
        return output

In [16]:
# test
model=Entity_Representation(768,1024)
x=torch.randn(32,10,768)
y=model(x)
print(x.shape)
print(y.shape)

torch.Size([32, 10, 768])
torch.Size([32, 10, 768])


In [45]:
# Span FFN
class Span_Representation(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()
        self.net=Sequential(
            Linear(2*d_model,d_ff),
            ReLU(),
            Linear(d_ff,d_model)
        )
    
    def forward(self,start_token,end_token):
        span=torch.cat([start_token,end_token],dim=-1)
        output=self.net(span)
        return output

In [18]:
# test
model=Span_Representation(768,1024)
start_token=torch.randn(32,20,768)
end_token=torch.randn(32,20,768)
y=model(start_token,end_token)
print("start shape : ",start_token.shape)
print("end shape : ",end_token.shape)
print(y.shape)

start shape :  torch.Size([32, 20, 768])
end shape :  torch.Size([32, 20, 768])
torch.Size([32, 20, 768])


In [19]:
class Sigmoid(nn.Module):
    def forward(self,x):
        output=1/(1+torch.exp(-x))
        return output

In [20]:
class Matching(nn.Module):
    def __init__(self):
        super().__init__()
        self.sigmoid=Sigmoid()
    def forward(self,entity,span):
        entity=entity.transpose(-1,-2)
        output=span@entity
        score=self.sigmoid(output)
        return score

In [21]:
# test
B,E,N,d=2,5,7,4
entity=torch.randn(B,E,d)
span=torch.randn(B,N,d)
model=Matching()
score=model(entity,span)
print(score.shape)
print(score)

torch.Size([2, 7, 5])
tensor([[[0.8350, 0.1991, 0.6854, 0.0769, 0.6614],
         [0.9359, 0.1968, 0.9200, 0.9727, 0.1793],
         [0.9617, 0.5088, 0.9443, 0.4259, 0.8250],
         [0.0426, 0.5802, 0.0491, 0.4992, 0.2088],
         [0.0670, 0.7938, 0.5499, 0.3504, 0.8960],
         [0.0381, 0.9716, 0.2505, 0.2219, 0.9447],
         [0.0200, 0.9269, 0.0073, 0.0160, 0.6790]],

        [[0.2867, 0.9488, 0.7517, 0.7344, 0.3093],
         [0.1140, 0.3722, 0.6398, 0.8211, 0.2111],
         [0.9272, 0.0923, 0.3393, 0.0624, 0.8333],
         [0.1353, 0.2245, 0.2499, 0.8690, 0.3234],
         [0.1674, 0.3687, 0.2229, 0.9057, 0.3602],
         [0.1026, 0.1492, 0.3357, 0.9313, 0.2528],
         [0.5216, 0.6434, 0.7694, 0.2681, 0.4370]]])


In [22]:
class BCEloss(nn.Module):
    def forward(self,pred,ans):
        eps=1e-12
        pred=torch.clamp(pred,eps,1-eps)
        loss=-(ans*torch.log(pred)+(1-ans)*torch.log(1-pred))
        output=loss.mean()
        return output

In [23]:
# test
target=torch.randint(0,2,(B,N,E)).float()
loss_f=BCEloss()
loss=loss_f(score,target)
print(loss)

tensor(0.8986)


In [ ]:
class Decoder(nn.Module):
    def __init__(self,entity_types,threshold=0.5,mode="flat"):
        super().__init__()
        self.entity_types=entity_types
        self.threshold=threshold
        self.mode=mode
    
    def overlapping(self,span1,span2):
        s1,e1=span1
        s2,e2=span2
        
        return not(e1<s2 or e2<s1)
    
    def partial_overlap(self,span1,span2):
        s1,e1=span1
        s2,e2=span2
        
        overlap=not(e1<s2 or e2<s1)
        if not overlap:
            return False
        
        nested=(s1<=s2 and e2<=e1) or (s2<=s1 and e1 <=e2)
        
        return not nested
    
    def decode_one(self,scores,span_indices):
        candiates=[]
        
        N_span,E=scores.shape
        
        for n in range(N_span):
            for e in range(E):
                score=scores[n, e].item()
                
                if score>self.threshold:
                    start,end=span_indices[n]
                    label=self.entity_types[e]
                    candiates.append((start,end,label,score))
        
        candiates.sort(key=lambda x:x[3],reverse=True)
        
        selected=[]
        
        for cand in candiates:
            c_start,c_end,c_label,c_score=cand
            conflict=False
            
            for sel in selected:
                s_start,s_end,_,_=sel
                
                if self.mode=="flat":
                    if self.overlapping((c_start,c_end),(s_start,s_end)):
                        conflict=True
                        
                        break
                    
                elif self.mode=="nested":
                    if self.partial_overlap((c_start,c_end),(s_start,s_end)):
                        conflict=True
                        
                        break
                    
            if not conflict:
                selected.append(cand)
                
        return selected
    
    def decode_batch(self,scores,span_indices):
        results=[]
        B=scores.shape[0]
        
        for b in range(B):
            results.append(self.decode_one(scores[b],span_indices))
            
        return results

In [40]:
# test
decoder=Decoder(
    entity_types=["PER","ORG"],
    threshold=0.5,
    mode="flat"
)

scores=torch.tensor([
    [0.9,0.1],
    [0.8,0.2],
    [0.3,0.7],
    [0.6,0.4],
],dtype=torch.float32)

span_indices=[
    (0,0),
    (0,1),
    (2,2),
    (1,2),
]
print(scores.shape)
result=decoder.decode_one(scores,span_indices)
print(result)

torch.Size([4, 2])
[(0, 0, 'PER', 0.8999999761581421), (2, 2, 'ORG', 0.699999988079071)]


In [41]:
# test
assert len(result)==2
assert result[0][0:3]==(0,0,"PER")
assert result[1][0:3]==(2,2,"ORG")
print("pass")

pass


In [42]:
# test
scores_batch = torch.tensor([
    [
        [0.9, 0.1],
        [0.8, 0.2],
        [0.3, 0.7],
        [0.6, 0.4],
    ],
    [
        [0.2, 0.9],
        [0.1, 0.4],
        [0.7, 0.2],
        [0.3, 0.8],
    ]
], dtype=torch.float32)

decoder = Decoder(
    entity_types=["PER", "ORG"],
    threshold=0.5,
    mode="flat"
)

results = decoder.decode_batch(scores_batch, span_indices)
print(results)

[[(0, 0, 'PER', 0.8999999761581421), (2, 2, 'ORG', 0.699999988079071)], [(0, 0, 'ORG', 0.8999999761581421), (1, 2, 'ORG', 0.800000011920929)]]


In [43]:
decoder_nested = Decoder(
    entity_types=["PER", "ORG"],
    threshold=0.5,
    mode="nested"
)

scores = torch.tensor([
    [0.95, 0.1],   # (0,2) -> PER
    [0.85, 0.2],   # (1,1) -> PER   : (0,2) 안에 완전 포함
    [0.80, 0.1],   # (1,3) -> PER   : (0,2)와 partial overlap
], dtype=torch.float32)

span_indices = [
    (0, 2),
    (1, 1),
    (1, 3),
]

result_nested = decoder_nested.decode_one(scores, span_indices)
print(result_nested)

[(0, 2, 'PER', 0.949999988079071), (1, 1, 'PER', 0.8500000238418579)]


In [ ]:
class GLiNER(nn.Module):
    def __init__(self,backbone_model,entity_types,d_ff,max_span_width=10,threshold=0.5,mode="flat"):
        super().__init__()
        
        self.entity_types=entity_types
        self.num_entity_types=len(entity_types)
        self.max_span_width=max_span_width
        
        self.encoder=EncoderModule(backbone_model)
        d_model=self.encoder.hidden_size
        
        self.text_token_rep=Text_Token_Represntation()
        self.ent_token_rep=ENT_Token_Representation()
        
        self.ent_rep=Entity_Representation(d_model,d_ff)
        self.span_rep=Span_Representation(d_model,d_ff)
        
        self.matching=Matching()
        
        self.decoder=Decoder(
            entity_types=entity_types,
            threshold=threshold,
            mode=mode
        )
        
    def build_span_representation(self,text_embeddings):
        B,T,H=text_embeddings.shape
        
        span_embeddings=[]
        span_indices=[]
        
        for start in range(T):
            max_end=min(T,start+self.max_span_width)
            for end in range(start,max_end):
                start_token=text_embeddings[:,start,:]
                end_token=text_embeddings[:,end,:]
                
                span_emb=self.span_rep(start_token,end_token)
                span_embeddings.append(span_emb)
                span_indices.append((start,end))
                
        span_embeddings=torch.stack(span_embeddings,dim=1)
        
        return span_embeddings,span_indices
        
    def forward(self,input_ids,attention_mask,text_mask,ent_mask):
        hidden_states=self.encoder(input_ids,attention_mask)
        
        text_embeddings=self.text_token_rep(hidden_states,text_mask)
        ent_embeddings=self.ent_token_rep(hidden_states,ent_mask)
        
        ent_vectors=self.ent_rep(ent_embeddings)
        span_vectors,span_indices=self.build_span_representation(text_embeddings)
        scores=self.matching(ent_vectors,span_vectors)
        return scores,span_indices
    
    @torch.no_grad()
    def predict(self,input_ids,attention_mask,text_mask,ent_mask):
        scores,span_indices=self.forward(input_ids=input_ids,attentio=attention_mask,text_mask=text_mask,ent_mask=ent_mask)
        
        results=self.decoder.decode_batch(scores,span_indices)
        
        return results

In [50]:
model=GLiNER(
    backbone_model=BackBone_Model,
    entity_types=["person","location","organization"],
    d_ff=512,
    max_span_width=10,
    threshold=0.5,
    mode="flat"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [54]:
scores, span_indices = model(input_ids, attention_mask, text_mask, ent_mask)
print(scores.shape)
print(span_indices[:5])

NameError: name 'input_ids' is not defined

In [58]:
import json
with open("train_gliner-3.jsonl","r",encoding='utf-8') as f:
    sample=json.loads(f.readline())

FileNotFoundError: [Errno 2] No such file or directory: 'train_gliner-3.jsonl'